# Import

In [41]:
import math
import torch
import torch.nn as nn

Input

|

MultiheadAttention

|

Add(Residual) (just adding x + attention_output) 

why?

if we replace x with new attention_output we will lose the values 

so update it 

|

LayerNorm (just a normalization layer)

In [42]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        
        # assert is keyword
        # its a debugging tools used to test if a specific condition in our code evalutes to true
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads


        self.Wq = nn.Linear(in_features = d_model, out_features = d_model)
        self.Wk = nn.Linear(in_features = d_model, out_features = d_model)
        self.Wv = nn.Linear(in_features = d_model, out_features = d_model)

        self.out_proj = nn.Linear(in_features = d_model, out_features = d_model)

    def forward(self, x):

        batch_size, seq_len, d_model = x.shape

        print('=' * 10)
        print("x shape ", x.shape)

        #send x in linear layer

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        print("\n After Linear Layers")

        print("Q : ", Q.shape)
        print("K : ", K.shape)
        print("V : ", V.shape)

        # split into heads using view
        # view is like reshaping but without copy 

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )
        

        print("\n After reshaping")
        print(Q.shape)
        print(K.shape)
        print(V.shape)

        # Move heads forward
        
        # change shape 
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        print("\n After Transpose")
        print("Q : ", Q.shape)
        print("K : ", K.shape)
        print("V : ", V.shape)

        # Attention score

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)

        print("\nscore")
        print(scores.shape)

        # softmax

        weights = torch.softmax(scores, dim = -1)

        print("\nAfter softmax Weight Shape")
        print(weights.shape)

        print("\nHead 1 Attentation matrix Weights")
        print(weights[0, 0])
        print("\nHead 2 Attentation matrix Weights")
        print(weights[0, 1])
        
        # Apply Attention
        output = weights @ V

        print("\nAfter Weights @ V")
        print(output.shape)

        # Combine heads
        output = output.transpose(1, 2)

        print("\nAfter Transpose Back")
        print(output.shape)

        # After Transpose we can't use view to reshape it 
        # so contiguous allocates a new block of memory to copy and rearrange a tensor's data into a sequential, unbroken memory layout.
        # then apply view()

        # print(output.contiguous())

        output = output.contiguous().view(
            batch_size,
            seq_len,
            d_model
        )

        # After Flatten
        print("\nAfter Flatten heads (2, 4 to 8)")
        print(output.shape)

        # Final projection

        output = self.out_proj(output)

        print("\nFinal output")
        print(output.shape)

        return output 

In [ ]:
class TransformerEncoderBlock(nn.Module):
    
    def __init__(self, d_model, num_heads):

        super().__init__()

        # created a obj for MHA
        self.mha = MultiHeadAttention(d_model = d_model, num_heads = num_heads)

        # LayerNorm from nn (new thing in V5)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):

        print("\n Input x")
        print(x)

        # giving value for the obj
        attention_output = self.mha(x)

        print("\nAttention output")
        print(attention_output.shape)

        # residual
        x += attention_output # Residual (new thing in V5)

        print("\nresidual output")
        print(x.shape)

        # normalization
        x = self.norm(x)

        print("\nNormalization (LayerNorm)")
        print(x.shape)

        return x



In [ ]:
# Example input

batch_size = 1
seq_len = 3
d_model = 8
num_heads = 2

vocab = {
    "I" : 0,
    "Love" : 1,
    "AI" : 2
}

tokens = torch.tensor([0, 1, 2]) # or torch.tensor(vocab.values())

embedding = nn.Embedding(
    num_embeddings = len(vocab),
    embedding_dim = d_model
)

x  = embedding(tokens)

x = x.view(1, 3, 8)


block = TransformerEncoderBlock(d_model = d_model, num_heads = num_heads)

output = block(x)

print("\noutput")
print(output.shape)


 Input x
tensor([[[-1.1954, -0.4945, -2.0717, -0.4180, -0.8784, -1.5308, -0.6301,
           0.2309],
         [-1.3220,  0.1208, -0.1831, -0.5541, -0.4194,  2.0167, -0.8725,
           1.4680],
         [-0.1546,  0.0788, -1.1680, -0.5527, -0.6753, -0.3323,  0.4742,
          -0.2491]]], grad_fn=<ViewBackward0>)
x shape  torch.Size([1, 3, 8])

 After Linear Layers
Q :  torch.Size([1, 3, 8])
K :  torch.Size([1, 3, 8])
V :  torch.Size([1, 3, 8])

 After reshaping
torch.Size([1, 3, 2, 4])
torch.Size([1, 3, 2, 4])
torch.Size([1, 3, 2, 4])

 After Transpose
Q :  torch.Size([1, 2, 3, 4])
K :  torch.Size([1, 2, 3, 4])
V :  torch.Size([1, 2, 3, 4])

score
torch.Size([1, 2, 3, 3])

After softmax Weight Shape
torch.Size([1, 2, 3, 3])

Head 1 Attentation matrix Weights
tensor([[0.3743, 0.2888, 0.3369],
        [0.4199, 0.1987, 0.3814],
        [0.3133, 0.3511, 0.3356]], grad_fn=<SelectBackward0>)

Head 2 Attentation matrix Weights
tensor([[0.2803, 0.3910, 0.3287],
        [0.1484, 0.6046, 0.247

In [45]:
print(x.mean())
print(x.std())


print(output.mean())
print(output.std())

tensor(-0.4570, grad_fn=<MeanBackward0>)
tensor(0.9031, grad_fn=<StdBackward0>)
tensor(9.9341e-09, grad_fn=<MeanBackward0>)
tensor(1.0215, grad_fn=<StdBackward0>)
